# Dialect-aware chrF Score with filtered n-grams

Which n-gram occurrences should be allowed to contribute to the score at all?

1. Extracts character n-grams (orders 1-6) directly from the unmodified ref/hyp sentences
2. Discards any n-gram that does not touch a dialect-specific character position (n-grams made up entirely of dialect-invariant characters are excluded)
3. Computes precision/recall on the filtered n-grams per order
4. Averages precision across orders and recall across orders separately, and then combine them into one F-beta score

* The dialect-specific / dialect-invariant mask itself is reused from Solution 2's NK-SK-hyp alignment logic

In [1]:
import string
import pandas as pd
from difflib import SequenceMatcher
from collections import Counter

In [2]:
PUNCT = set(string.punctuation)
NGRAM_ORDER = 6   # same default as sacrebleu's CHRF
BETA = 3          # chrF3 -> recall weighted 3x than precision

## Dialect mask construction

Dialect-specific = True, Dialect-invariant = False

### Locate candidate invariant positions from NK–SK reference

code reused from solution 2

In [3]:
def mark_range(mask: list[bool], start: int, end: int, value: bool) -> None:
    """Assign one label to all positions in [start, end)."""

    for index in range(start, end):
        mask[index] = value
    # True : dialect-invariant, False : not dialect-invariant(dialect-specific)
    
    # SK reference
    # 책 상  밑 으 로
    # 0 1 2 3 4 5
    # a mask might be : [True, True, False, True, True, True]

In [4]:
def merge_overlapping_spans(spans):
    """
    Merge overlapping or adjacent character spans.
    
    Different spacing edits may expand into overlapping regions!
    
    Example : [(5, 10), (8, 14), (20, 25)] becomes [(5, 14), (20, 25)]
    
    """

    # if no dialect-specific spans were found return empty list (safety check)
    if not spans:
        return []

    # sort dialect-specific spans in left-to-right order
    spans = sorted(spans)
    
    # initialize the merged list with the first span
    merged = [spans[0]]

    # compare each remaining span with the most recently merged span
    for start, end in spans[1:]:
        previous_start, previous_end = merged[-1]

        # if the current span overlaps with the previous span, combine them into a single larger span
        if start <= previous_end:
            merged[-1] = (previous_start, max(previous_end, end))

        else:
            merged.append((start, end))
    
    # return a set of non-overlapping dialect-specific spans
    return merged

In [5]:
def expand_spacing_region(text: str, start: int, end: int) -> tuple[int, int]:
    """
    Expand a whitespace edit to the expressions immediately surrounding the spacing boundary
    so that the spacing difference contributes to the dialect-specific score.
    """

    # move left until reaching either the beginning of the sentence or the whitespace preceding the expression
    while start > 0 and not text[start - 1].isspace():
        start -= 1
        
    # continue moving right until reaching the next whitespace boundary
    while end < len(text) and not text[end].isspace():
        end += 1

    return start, end

In [6]:
def build_initial_invariant_mask(nk: str, sk: str) -> list[bool]:
    """
    Identify candidate invariant positions in the SK reference.

    #1. Mark characters aligned in NK-SK equal blocks as invariant

    #2. Remove spacing-sensitive regions from the invariant set so that
        NK-SK spacing differences contribute to the dialect-specific score
    """
    
    matcher = SequenceMatcher(None, nk, sk, autojunk=False)

    # False means dialect-specific; True means dialect-invariant
    invariant_mask = [False] * len(sk)
    # store SK-reference regions associated with spacing differences
    spacing_regions = []

    for tag, nk_start, nk_end, sk_start, sk_end in matcher.get_opcodes():
        # text covered by the current opcode on each side
        nk_text = nk[nk_start:nk_end]
        sk_text = sk[sk_start:sk_end]

        # characters in an equal block are identical between the NK and SK references,
        # and initially considered candidate dialect-invariant characters
        # mark the corresponding character positions in the SK reference as True
        if tag == "equal":
            mark_range(invariant_mask, sk_start, sk_end, True)
            continue

        # if a whitespace character exists in SK but not in NK
        # the inserted whitespace has an explicit character span on the SK side,
        # expand from this whitespace span in both directions to capture the surrounding expression
        if tag == "insert" and sk_text and sk_text.isspace():
            spacing_regions.append(expand_spacing_region(sk, sk_start, sk_end))
            continue

        # if a whitespace character exists in NK but not in SK
        # there is no corresponding SK character for the deleted space,
        # so sk_start represents its corresponding boundary in the SK sentence
        # expand from this boundary to both sides to capture the surrounding expression        
        if tag == "delete" and nk_text and nk_text.isspace():
            spacing_regions.append(expand_spacing_region(sk, sk_start, sk_start))
            continue

        # handles cases where SequenceMatcher represents a spacing change as a replace rather than a clean insertion/deletion
        if (tag == "replace" and nk_text.replace(" ", "") == sk_text.replace(" ", "") and nk_text != sk_text):
            spacing_regions.append(expand_spacing_region(sk, sk_start, sk_end))

    # since several spacing edits may expand into overlapping regions, merge them before changing the invariant mask
    for start, end in merge_overlapping_spans(spacing_regions):
        # spacing-sensitive regions must not remain invariant
        mark_range(invariant_mask, start, end, False)

    return invariant_mask

### Among the candidate invariant characters keep only positions that also remain equal in the hypothesis

code reused from solution 2

In [7]:
def refine_invariant_mask_with_hypothesis(ref: str, hyp: str, initial_ref_invariant_mask: list[bool]) -> tuple[list[bool], list[bool]]:
    """
    Keep text invariant only when it is both:

    1. initially invariant between the NK and SK references
    2. unchanged between the SK reference and SK hypothesis

    Any reference-hypothesis difference is excluded from the invariant
    partition and therefore becomes part of the dialect-specific complement.
    """
    
    matcher = SequenceMatcher(None, ref, hyp, autojunk=False)

    final_ref_invariant_mask = [False] * len(ref)
    hyp_invariant_mask = [False] * len(hyp)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag != "equal":
            # replace, insert, and delete regions cannot be invariant because the reference and hypothesis differ there
            continue

        # equal blocks have direct one-to-one character correspondence
        for offset in range(i2 - i1):
            ref_index = i1 + offset
            hyp_index = j1 + offset

            # a character is finally invariant only if 
            # 1. it was invariant in the NK–SK comparison
            # 2. it belongs to an equal SK-reference–hypothesis block
            if initial_ref_invariant_mask[ref_index]:
                final_ref_invariant_mask[ref_index] = True
                hyp_invariant_mask[hyp_index] = True

    return final_ref_invariant_mask, hyp_invariant_mask

### treat the complement of dialect-invariant mask = dialect-specific mask

In [8]:
def build_dialect_masks(nk: str, sk: str, hyp: str) -> tuple[list[bool], list[bool]]:
    """
    1. Identify candidate invariant characters from the NK-SK pair (+ remove spacing-sensitive regions from that invariant set)
    2. Retain only invariant characters that are also unchanged in the SK reference and hypothesis
    3. Treat the complement as dialect-specific
    """
    
    # 1. locate candidate invariant positions from NK–SK reference
    initial_mask = build_initial_invariant_mask(nk, sk)
    
    # 2. keep only positions that also remain equal in the hypothesis
    ref_invariant_mask, hyp_invariant_mask = refine_invariant_mask_with_hypothesis(sk, hyp, initial_mask)
    
    # 3. dialect masks = negation of invariant masks (characters that are True (dialect-invariant) become False and vice versa)
    ref_dialect_mask = [not c for c in ref_invariant_mask]
    hyp_dialect_mask = [not c for c in hyp_invariant_mask]
    
    # returns (ref_dialect_mask, hyp_dialect_mask): True = dialect-specific character, False = dialect-invariant character
    return ref_dialect_mask, hyp_dialect_mask

## Filtered n-gram extraction + chrF computation

In [9]:
def extract_char_ngrams_with_positions(text: str, n: int):
    # extract all length-n character windows of 'text', along with their [start, end) offsets
    return [(text[i:i + n], i, i + n) for i in range(len(text) - n + 1)]

In [ ]:
def filter_ngrams_by_mask(ngrams_with_positions, mask: list[bool]):
    """
    Keep only n-grams whose window overlaps parts where 'mask' is True (dialect-specific)
    
    e.g., 2-grams
    hyp_all_ngrams = [('나는', 0, 2), ('는_', 1, 3), ... , ('갔다', 41, 43), ('다.', 42, 44)]
    hyp_mask = [F, ... , F, T, F, ... , F, T, ... , T, F, F, T, F, ... , F, T, ... , T, F, ... , F]
    hyp_filtered = ['뛰어', '어넘', '_책', '책상', '상_', '_밑', '밑으', '으로', '로_', '기여', '여나', '_난', '난로', '로_', '_옆', '옆까', '까지', '지_']
    ref_filtered = ['뛰어', '어넘', '_책', '책상', '상_', '_밑', '밑으', '으로', '로_', '기어', '어나', '_난', '난로', '로_', '_옆', '옆까', '까지', '지_']
    """
    kept = [] # a list to save n-grams that include at least one dialect-specific charaacter
    
    for ngram, start, end in ngrams_with_positions:
        # slice out just the portion of the mask that lines up with this n-gram's window
        window = mask[start:end] 
        # how many characters in this window are dialect-specific
        dialect_count = sum(window)
        # length of the window
        window_len = end - start
        
        # keep the n-gram if it touches at least one dialect-specific character, even if the rest of the characters in the window is dialect-invariant
        keep = dialect_count >= 1
        
        # only n-grams that pass the chosen rule survive into the filtered
        # list that clipped_match_count() will later compare against
        if keep:
            kept.append(ngram)
            
    return kept

In [11]:
def clipped_match_count(hyp_ngrams: list[str], ref_ngrams: list[str]) -> int:
    """
    clipping makes sure the score reflects genuine overlap with the reference, not just raw repetition of strings in the hypothesis
    """
    # for each n-gram TYPE, take min(count in hyp, count in ref), then sum to find the number of matching characters
    
    # e.g., n = 2
    hyp_counter = Counter(hyp_ngrams)
    # {뛰어:1, 어넘:1, _책:1, 책상:1, 상_:1, _밑:1, 밑으:1, 으로:1, 로_:2, 기여:1, 여나:1, _난:1, 난로:1, _옆:1, 옆까:1, 까지:1, 지_:1}

    ref_counter = Counter(ref_ngrams)
    # {뛰어:1, 어넘:1, _책:1, 책상:1, 상_:1, _밑:1, 밑으:1, 으로:1, 로_:2, 기어:1, 어나:1, _난:1, 난로:1, _옆:1, 옆까:1, 까지:1, 지_:1}
    
    common = hyp_counter & ref_counter 
    # {뛰어:1, 어넘:1, _책:1, 책상:1, 상_:1, _밑:1, 밑으:1, 으로:1, 로_:2, _난:1, 난로:1, _옆:1, 옆까:1, 까지:1, 지_:1}
    
    # adds up all those clipped per-type counts into a single total match count
    return sum(common.values()) # 3

In [12]:
def filtered_chrf(hyp: str, ref: str, hyp_mask: list[bool], ref_mask: list[bool], max_order: int = NGRAM_ORDER, beta: int = BETA,) -> float:
    """
    Dialect-filtered chrF score, computed on the FULL sentence rather than computing separate chrF scores for the dialect-specific parts and dialect-invariant parts
    """
    precisions, recalls = [], []
    
    # for each order n = 1..6:
    for n in range(1, max_order + 1):
        # extract every character n-gram from the full ref / hyp strings
        hyp_all_ngrams = extract_char_ngrams_with_positions(hyp, n)
        ref_all_ngrams = extract_char_ngrams_with_positions(ref, n)

        # keeps only the n-grams that overlap a dialect-specific character
        hyp_filtered = filter_ngrams_by_mask(hyp_all_ngrams, hyp_mask)
        ref_filtered = filter_ngrams_by_mask(ref_all_ngrams, ref_mask)
 
        total_hyp = len(hyp_filtered)
        total_ref = len(ref_filtered)
 
        if total_hyp == 0 and total_ref == 0:
            continue  # nothing to compare at this order; skip rather than force 0
        
        # compute precision and recall on the surviving n-grams to compute final chrF
        
        # of the dialect-relevant n-grams hyp produced, how many genuinely overlap with what ref needed?
        matched = clipped_match_count(hyp_filtered, ref_filtered)

        # precision for each order
        p_n = matched / total_hyp if total_hyp > 0 else 0.0 # of what hyp produced, how much was actually right?
        
        # recall for each order
        r_n = matched / total_ref if total_ref > 0 else 0.0 # of what ref needed, how much did hyp actually produce?

        # keep precision and recall values for each order
        precisions.append(p_n)
        recalls.append(r_n)
 
    if not precisions:
        return 0.0  # no dialect-relevant n-grams survived at any order
    
    # the final character precision and recall obtained by averaging across all n-gram orders
    mean_p = sum(precisions) / len(precisions)
    mean_r = sum(recalls) / len(recalls)
 
    if mean_p == 0.0 and mean_r == 0.0:
        return 0.0
    
    # chrF score
    chrf_final = ((1 + beta ** 2) * mean_p * mean_r) / (beta ** 2 * mean_p + mean_r)
    return round(chrf_final * 100, 2)

##  Evaluate each sentence pair

In [13]:
def evaluate_one_example(nk: str, sk: str, hyp: str):
    """
    Compute one dialect-aware chrF3 score per example 
    """
    
    # 1. Create dialect-specific mask
    ref_dialect_mask, hyp_dialect_mask = build_dialect_masks(nk, sk, hyp)
    
    # 2. n-grams are extracted from the full sk/hyp sentences and only those that include at least one dialect-specific character
    dialect_aware_score = filtered_chrf(hyp, sk, hyp_dialect_mask, ref_dialect_mask)

    return {"dialect_aware_chrF3_filtered": dialect_aware_score}

In [14]:
def evaluate_outputs(ref_path, hyp_path, save_path):
    ref = pd.read_csv(ref_path, sep="\t")
    ref = ref.rename(columns={"NK": "nk", "SK": "sk"})
    ref.insert(0, "id", range(1, len(ref) + 1))
 
    hyp = pd.read_csv(hyp_path, sep="\t")
    df = ref.merge(hyp, on="id", how="left")
 
    rows = []
    for _, row in df.iterrows():
        result = evaluate_one_example(
            nk=str(row["nk"]), sk=str(row["sk"]), hyp=str(row["hyp"])
        )
        rows.append({
            "id": row["id"],
            "nk": row["nk"],
            "ref_sk": row["sk"],
            "hyp_sk": row["hyp"],
            **result,
        })
 
    result_df = pd.DataFrame(rows)
    result_df.to_csv(save_path, sep="\t", index=False)
 
    summary = result_df[["dialect_aware_chrF3_filtered"]].mean()
 
    print(f"\n===== Mean Sentence-level Results =====")
    print(summary)
 
    return result_df, summary
 
 
if __name__ == "__main__":
    evaluate_outputs(
        ref_path="../data/input/minimal_pairs_sample_ref.tsv",
        hyp_path="../data/input/minimal_pairs_sample_hyp.tsv",
        save_path="../data/output/dialect_aware_chrf_ngrams_filtered_results.tsv"
    )


===== Mean Sentence-level Results =====
dialect_aware_chrF3_filtered    47.7425
dtype: float64
